# Edge IIoT — Binary Classification Results Aggregation

Loads all trained models from `models/binary/edge_iiot/`, reconstructs the feature sets for each experiment, and produces a unified comparison CSV at `results/binary_classification_results.csv`.

**Requirements:** All experiment notebooks must have been run at least once so their model `.pkl` files exist.

In [1]:
%load_ext autoreload
%autoreload 2

import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score
)

from src.config import DATASETS, SEED

dataset_name = "edge_iiot"
config = DATASETS[dataset_name]

## 1. Load Dataset

In [2]:
pkl_path = config['processed_path'] / 'ML-EdgeIIoT-dataset.pkl'
print(f"Loading dataset from: {pkl_path}")
df = pd.read_pickle(pkl_path)
print(f"Dataset shape: {df.shape}")

Loading dataset from: /home/uo294319/ML-NIDS-IIoT/data/edge_iiot/processed/ML-EdgeIIoT-dataset.pkl
Dataset shape: (152590, 62)


## 2. Build Feature Sets

Reconstruct the exact feature engineering pipeline for each experiment so that saved models receive correctly processed inputs.

In [3]:
target = 'Attack_label'
Y_str = df['Attack_type']

# --- Shared base ---
df_numeric = df.select_dtypes(include=['number'])
X_base = df_numeric.drop(columns=[target])
y = df_numeric[target]

# --- Experiment 1: all numeric features (50) ---
X_exp1 = X_base.copy()
print(f"Exp1 feature shape: {X_exp1.shape}")

# --- Experiment 2: drop environment-specific features (40) ---
cols_to_drop_exp2 = [
    'frame.time.delta', 'frame.time.order',
    *[c for c in X_base.columns if c.startswith('ip.src_category') or c.startswith('ip.dst_category')]
]
X_exp2 = X_base.drop(columns=cols_to_drop_exp2, errors='ignore')
print(f"Exp2 feature shape: {X_exp2.shape}")

Exp1 feature shape: (152590, 53)
Exp2 feature shape: (152590, 43)


## 3. Reproduce Train/Test Splits

In [4]:
def make_split(X, y, Y_str, random_state=SEED, test_size=0.2):
    """Reproduce the exact split used during training."""
    _, X_test, _, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=Y_str
    )
    return X_test, y_test

X_test_exp1, y_test_exp1 = make_split(X_exp1, y, Y_str)
X_test_exp2, y_test_exp2 = make_split(X_exp2, y, Y_str)

print(f"Exp1 test shape: {X_test_exp1.shape}")
print(f"Exp2 test shape: {X_test_exp2.shape}")

Exp1 test shape: (30518, 53)
Exp2 test shape: (30518, 43)


## 4. Evaluate All Saved Models

In [5]:
model_base = config['bin_model_path']

# Map experiment name -> (X_test, y_test)
experiment_data = {
    'exp1': (X_test_exp1, y_test_exp1),
    'exp2': (X_test_exp2, y_test_exp2),
}

# Map model filename stem -> display name
model_display = {
    'model_xgb':  'XGBoost',
    'model_et':   'ExtraTrees',
    'model_lgbm': 'LightGBM',
}

records = []

for exp_name, (X_test, y_test) in experiment_data.items():
    exp_path = model_base / exp_name
    if not exp_path.exists():
        print(f"[SKIP] {exp_path} not found — run the experiment notebooks first.")
        continue

    for model_stem, model_display_name in model_display.items():
        model_file  = exp_path / f"{model_stem}.pkl"
        scaler_stem = model_stem.replace('model_', 'scaler_')
        scaler_file = exp_path / f"{scaler_stem}.pkl"

        if not model_file.exists():
            print(f"[SKIP] {model_file.name} not found in {exp_path}")
            continue

        model  = joblib.load(model_file)
        scaler = joblib.load(scaler_file) if scaler_file.exists() else None

        if scaler is not None:
            X_scaled = scaler.transform(X_test)
        else:
            X_scaled = X_test.values

        y_pred = model.predict(X_scaled)
        y_prob = model.predict_proba(X_scaled)[:, 1] if hasattr(model, 'predict_proba') else y_pred

        records.append({
            'experiment':          exp_name,
            'model':               model_display_name,
            'n_features':          X_test.shape[1],
            'accuracy':            round(accuracy_score(y_test, y_pred), 6),
            'f1_weighted':         round(f1_score(y_test, y_pred, average='weighted'), 6),
            'f1_macro':            round(f1_score(y_test, y_pred, average='macro'), 6),
            'roc_auc':             round(roc_auc_score(y_test, y_prob), 6),
            'precision_weighted':  round(precision_score(y_test, y_pred, average='weighted'), 6),
            'recall_weighted':     round(recall_score(y_test, y_pred, average='weighted'), 6),
        })
        print(f"[OK] {exp_name} / {model_display_name}: acc={records[-1]['accuracy']:.4f}  f1_w={records[-1]['f1_weighted']:.4f}")

results_df = pd.DataFrame(records)
print(f"\nCollected {len(results_df)} results.")

[OK] exp1 / XGBoost: acc=0.9989  f1_w=0.9989
[OK] exp1 / ExtraTrees: acc=0.9989  f1_w=0.9989
[OK] exp1 / LightGBM: acc=0.9990  f1_w=0.9990
[OK] exp2 / XGBoost: acc=0.9746  f1_w=0.9737


/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] exp2 / ExtraTrees: acc=0.9748  f1_w=0.9739
[OK] exp2 / LightGBM: acc=0.9749  f1_w=0.9740

Collected 6 results.


/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 5. Save Results CSV

In [6]:
results_dir = Path(config['bin_model_path']).parent.parent.parent / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

out_path = results_dir / 'binary_classification_results.csv'
results_df.to_csv(out_path, index=False)
print(f"Results saved to: {out_path}")

Results saved to: /home/uo294319/ML-NIDS-IIoT/results/binary_classification_results.csv


## 6. Summary Table

In [7]:
metric_cols = ['accuracy', 'f1_weighted', 'f1_macro', 'roc_auc', 'precision_weighted', 'recall_weighted']

if results_df.empty:
    print("No results to display. Run the experiment notebooks first.")
else:
    styled = (
        results_df
        .sort_values(['experiment', 'f1_weighted'], ascending=[True, False])
        .reset_index(drop=True)
        .style
        .background_gradient(subset=metric_cols, cmap='YlGn')
        .format({col: '{:.4f}' for col in metric_cols})
        .set_caption('Binary Classification Results — Edge IIoTset')
    )
    display(styled)

,experiment,model,n_features,accuracy,f1_weighted,f1_macro,roc_auc,precision_weighted,recall_weighted
0,exp1,LightGBM,53,0.9990,0.9990,0.9982,1.0000,0.9990,0.9990
1,exp1,ExtraTrees,53,0.9989,0.9989,0.9980,1.0000,0.9989,0.9989
2,exp1,XGBoost,53,0.9989,0.9989,0.9979,1.0000,0.9989,0.9989
3,exp2,LightGBM,43,0.9749,0.9740,0.9498,0.9930,0.9756,0.9749
4,exp2,ExtraTrees,43,0.9748,0.9739,0.9496,0.9925,0.9755,0.9748
5,exp2,XGBoost,43,0.9746,0.9737,0.9493,0.9929,0.9753,0.9746


## 7. Exp1 vs Exp2 Delta

Measure the performance impact of removing environment-specific features.

In [8]:
if not results_df.empty and 'exp1' in results_df['experiment'].values and 'exp2' in results_df['experiment'].values:
    pivot = results_df.pivot_table(index='model', columns='experiment', values=metric_cols)
    pivot.columns = [f"{metric}_{exp}" for metric, exp in pivot.columns]
    pivot = pivot.reset_index()

    for metric in metric_cols:
        if f"{metric}_exp1" in pivot.columns and f"{metric}_exp2" in pivot.columns:
            pivot[f"{metric}_delta"] = (pivot[f"{metric}_exp2"] - pivot[f"{metric}_exp1"]).round(6)

    delta_cols = ['model'] + [f"{m}_delta" for m in metric_cols if f"{m}_delta" in pivot.columns]
    print("Performance delta (exp2 - exp1). Negative = features help; positive = features hurt.")
    display(pivot[delta_cols].style.background_gradient(subset=[c for c in delta_cols if 'delta' in c], cmap='RdYlGn').format({c: '{:+.4f}' for c in delta_cols if 'delta' in c}))
else:
    print("Need both exp1 and exp2 results to compute delta.")

Performance delta (exp2 - exp1). Negative = features help; positive = features hurt.


,model,accuracy_delta,f1_weighted_delta,f1_macro_delta,roc_auc_delta,precision_weighted_delta,recall_weighted_delta
0,ExtraTrees,-0.0242,-0.0250,-0.0484,-0.0074,-0.0234,-0.0242
1,LightGBM,-0.0242,-0.0251,-0.0484,-0.0070,-0.0235,-0.0242
2,XGBoost,-0.0243,-0.0252,-0.0486,-0.0071,-0.0236,-0.0243
